In [7]:
!pip -q install -U datasets transformers accelerate scikit-learn pandas numpy scipy tqdm matplotlib torchaudio
!pip -q install -U captum || true


ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
captum 0.8.0 requires numpy<2.0, but you have numpy 2.4.0 which is incompatible.
opencv-python 4.12.0.88 requires numpy<2.3.0,>=2; python_version >= "3.9", but you have numpy 2.4.0 which is incompatible.
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
opencv-python 4.12.0.88 requires numpy<2.3.0,>=2; python_version >= "3.9", but you have numpy 1.26.4 which is incompatible.


In [16]:
pip install -U datasets transformers accelerate scikit-learn pandas scipy tqdm matplotlib torchaudio


Note: you may need to restart the kernel to use updated packages.


In [12]:
pip uninstall -y captum


Found existing installation: captum 0.8.0
Uninstalling captum-0.8.0:
  Successfully uninstalled captum-0.8.0
Note: you may need to restart the kernel to use updated packages.


In [14]:
pip install -U "numpy>=2,<2.3"

  Using cached numpy-2.2.6-cp311-cp311-win_amd64.whl.metadata (60 kB)
Using cached numpy-2.2.6-cp311-cp311-win_amd64.whl (12.9 MB)
  Attempting uninstall: numpy
    Found existing installation: numpy 1.26.4
    Uninstalling numpy-1.26.4:
      Successfully uninstalled numpy-1.26.4
Note: you may need to restart the kernel to use updated packages.


  You can safely remove it manually.
  You can safely remove it manually.


In [15]:
pip install -U opencv-python

Note: you may need to restart the kernel to use updated packages.


In [18]:
pip install -U "datasets[audio]" torchcodec

   ---------------------------------------- 0.0/2.2 MB ? eta -:--:--
   ---------------------------------------- 0.0/2.2 MB ? eta -:--:--
   ---- ----------------------------------- 0.3/2.2 MB ? eta -:--:--
   -------------- ------------------------- 0.8/2.2 MB 2.4 MB/s eta 0:00:01
   ---------------------------------------- 2.2/2.2 MB 4.9 MB/s  0:00:00
Note: you may need to restart the kernel to use updated packages.


In [1]:
import os, re, math, json, time, random, warnings
from dataclasses import dataclass, asdict
from typing import Dict, List, Tuple, Optional, Any

import numpy as np
import pandas as pd

import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader

import torchaudio
from tqdm.auto import tqdm
import matplotlib.pyplot as plt

from datasets import load_dataset
from transformers import (
    DistilBertTokenizerFast,
    DistilBertModel,
    Wav2Vec2Model,
    Wav2Vec2Processor,
    get_linear_schedule_with_warmup,
)

from sklearn.metrics import accuracy_score, f1_score, recall_score, confusion_matrix, cohen_kappa_score
warnings.filterwarnings("ignore")

def set_seed(seed: int = 42):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    torch.cuda.manual_seed_all(seed)

set_seed(42)
DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
DEVICE


'cuda'

In [2]:
@dataclass
class Config:
    # HF dataset
    hf_dataset: str = "AbstractTTS/IEMOCAP"
    hf_split: str = "train"

    # Emotion labels to use:
    #   "major_emotion_10"  -> use dataset's major_emotion (10 classes)
    #   "iemocap_4"         -> map to {neu, hap(hap+exc), ang, sad}
    #   "iemocap_6"         -> {neu, hap, ang, sad, exc, fru} (if present)
    emotion_label_set: str = "iemocap_4"

    # Distress target (IEMOCAP has EmoAct 1..5; use as distress intensity)
    #   "activation_regression" (recommended)
    #   "activation_categorical" (bins -> low/med/high/extreme)
    #   "activation_ordinal" (CORAL)
    distress_target: str = "activation_regression"
    activation_bins: Tuple[float, float, float] = (2.5, 3.5, 4.5)  # cutpoints for 4 bins

    # Emergency-style segmentation (optional)
    use_sliding_window: bool = False
    segment_seconds: float = 6.0
    segment_overlap: float = 0.5

    # Audio normalization
    target_sr: int = 16000
    max_audio_seconds: float = 10.0  # truncate/pad after segmentation

    # Encoders
    wav2vec_name: str = "facebook/wav2vec2-base-960h"
    text_name: str = "distilbert-base-uncased"

    # Fusion dimensions
    proj_dim: int = 256
    num_heads: int = 4
    drb_use_stats: bool = True

    # Ablations
    fusion_mode: str = "fusion"  # "fusion" | "audio_only" | "text_only"

    # Training
    batch_size: int = 6
    num_epochs: int = 6
    lr: float = 2e-4
    weight_decay: float = 0.01
    warmup_ratio: float = 0.06
    grad_clip: float = 1.0
    mixed_precision: bool = True

    # Multi-task loss weights grid (as you wrote)
    lambda_grid: Tuple[float, ...] = (0.3, 0.5, 0.7)

    # Output
    out_dir: str = "./dersx_hf_iemocap_runs"

cfg = Config()
os.makedirs(cfg.out_dir, exist_ok=True)
cfg


Config(hf_dataset='AbstractTTS/IEMOCAP', hf_split='train', emotion_label_set='iemocap_4', distress_target='activation_regression', activation_bins=(2.5, 3.5, 4.5), use_sliding_window=False, segment_seconds=6.0, segment_overlap=0.5, target_sr=16000, max_audio_seconds=10.0, wav2vec_name='facebook/wav2vec2-base-960h', text_name='distilbert-base-uncased', proj_dim=256, num_heads=4, drb_use_stats=True, fusion_mode='fusion', batch_size=6, num_epochs=6, lr=0.0002, weight_decay=0.01, warmup_ratio=0.06, grad_clip=1.0, mixed_precision=True, lambda_grid=(0.3, 0.5, 0.7), out_dir='./dersx_hf_iemocap_runs')

In [3]:
ds = load_dataset(cfg.hf_dataset, split=cfg.hf_split)
print(ds)
print("Columns:", ds.column_names)
print("Features:", ds.features)


Dataset({
    features: ['file', 'audio', 'frustrated', 'angry', 'sad', 'disgust', 'excited', 'fear', 'neutral', 'surprise', 'happy', 'EmoAct', 'EmoVal', 'EmoDom', 'gender', 'transcription', 'major_emotion', 'speaking_rate', 'pitch_mean', 'pitch_std', 'rms', 'relative_db'],
    num_rows: 10039
})
Columns: ['file', 'audio', 'frustrated', 'angry', 'sad', 'disgust', 'excited', 'fear', 'neutral', 'surprise', 'happy', 'EmoAct', 'EmoVal', 'EmoDom', 'gender', 'transcription', 'major_emotion', 'speaking_rate', 'pitch_mean', 'pitch_std', 'rms', 'relative_db']
Features: {'file': Value('string'), 'audio': Audio(sampling_rate=None, decode=True, num_channels=None, stream_index=None), 'frustrated': Value('float32'), 'angry': Value('float32'), 'sad': Value('float32'), 'disgust': Value('float32'), 'excited': Value('float32'), 'fear': Value('float32'), 'neutral': Value('float32'), 'surprise': Value('float32'), 'happy': Value('float32'), 'EmoAct': Value('float32'), 'EmoVal': Value('float32'), 'EmoDom': 

In [4]:
def scrub_pii(text: str) -> str:
    # Basic PII scrubbing heuristics (customize as needed)
    text = re.sub(r"\b\d{3}[-.\s]?\d{3}[-.\s]?\d{4}\b", "[PHONE]", text)  # phone
    text = re.sub(r"\b\d{1,4}\s+\w+(\s+\w+){0,3}\s+(street|st|road|rd|ave|avenue|blvd|lane|ln)\b",
                  "[ADDRESS]", text, flags=re.IGNORECASE)
    text = re.sub(r"\b[\w\.-]+@[\w\.-]+\.\w+\b", "[EMAIL]", text)
    return text

def clean_text(text: str) -> str:
    text = (text or "").strip()
    text = scrub_pii(text)
    text = re.sub(r"\s+", " ", text)
    return text

def parse_session(file_name: str) -> str:
    # e.g. "Ses01F_impro01_F000.wav" -> "Ses01"
    m = re.match(r"(Ses\d{2})", file_name)
    return m.group(1) if m else "UNK"

def parse_dialog_id(file_name: str) -> str:
    # remove extension
    base = os.path.splitext(file_name)[0]
    parts = base.split("_")
    # drop last chunk (turn id like F000 / M003 etc)
    if len(parts) >= 2:
        return "_".join(parts[:-1])
    return base

def parse_speaker(file_name: str) -> str:
    # often Ses01F..., so speaker is char after Ses##.
    m = re.match(r"Ses\d{2}([FM])", file_name)
    return m.group(1) if m else "U"


In [5]:
# Convert to a light index DataFrame (do NOT expand audio arrays here)
df = pd.DataFrame({
    "idx": np.arange(len(ds)),
    "file": ds["file"] if "file" in ds.column_names else [None]*len(ds),
    "transcription": [clean_text(x) for x in ds["transcription"]] if "transcription" in ds.column_names else ["" for _ in range(len(ds))],
    "major_emotion": ds["major_emotion"] if "major_emotion" in ds.column_names else [None]*len(ds),
    "EmoAct": ds["EmoAct"] if "EmoAct" in ds.column_names else [None]*len(ds),
    "EmoVal": ds["EmoVal"] if "EmoVal" in ds.column_names else [None]*len(ds),
    "EmoDom": ds["EmoDom"] if "EmoDom" in ds.column_names else [None]*len(ds),
    "gender": ds["gender"] if "gender" in ds.column_names else [None]*len(ds),
})

df["session"] = df["file"].apply(lambda x: parse_session(x) if isinstance(x, str) else "UNK")
df["dialog_id"] = df["file"].apply(lambda x: parse_dialog_id(x) if isinstance(x, str) else "UNK")
df["speaker"] = df["file"].apply(lambda x: parse_speaker(x) if isinstance(x, str) else "U")

# Filter rows missing key fields
df = df[df["transcription"].str.len() > 0].copy()
df = df[df["EmoAct"].notnull()].copy()
df.reset_index(drop=True, inplace=True)

df.head(), df["session"].value_counts()


(   idx                     file  \
 0    0  Ses01F_impro01_F000.wav   
 1    1  Ses01F_impro01_F001.wav   
 2    2  Ses01F_impro01_F002.wav   
 3    3  Ses01F_impro01_F003.wav   
 4    4  Ses01F_impro01_F004.wav   
 
                                        transcription major_emotion    EmoAct  \
 0                                         Excuse me.       neutral  2.333333   
 1                                              Yeah.       neutral  2.666667   
 2                                Is there a problem?       neutral  2.666667   
 3                                           You did.       neutral  3.000000   
 4  You were standing at the beginning and you dir...       neutral  3.333333   
 
      EmoVal    EmoDom  gender session       dialog_id speaker  
 0  2.666667  2.000000  Female   Ses01  Ses01F_impro01       F  
 1  2.333333  2.333333  Female   Ses01  Ses01F_impro01       F  
 2  2.666667  2.666667  Female   Ses01  Ses01F_impro01       F  
 3  2.333333  3.000000  Female   S

In [6]:
def map_emotion(label: str, label_set: str) -> Optional[str]:
    if label is None:
        return None
    lab = str(label).lower().strip()

    if label_set == "major_emotion_10":
        return lab  # use as-is

    if label_set == "iemocap_4":
        # {neu, hap (happy+excited), ang, sad}
        if lab in ["neutral", "neu"]:
            return "neu"
        if lab in ["happy", "hap", "excited", "exc"]:
            return "hap"
        if lab in ["angry", "ang"]:
            return "ang"
        if lab in ["sad", "sadness"]:
            return "sad"
        return None

    if label_set == "iemocap_6":
        # {neu, hap, ang, sad, exc, fru}
        mapping = {
            "neutral": "neu", "neu": "neu",
            "happy": "hap", "hap": "hap",
            "angry": "ang", "ang": "ang",
            "sad": "sad",
            "excited": "exc", "exc": "exc",
            "frustrated": "fru", "fru": "fru",
        }
        return mapping.get(lab, None)

    raise ValueError("Unknown emotion_label_set")

df["emotion"] = df["major_emotion"].apply(lambda x: map_emotion(x, cfg.emotion_label_set))
df = df[df["emotion"].notnull()].reset_index(drop=True)

emo_classes = sorted(df["emotion"].unique().tolist())
emo2id = {e:i for i,e in enumerate(emo_classes)}
id2emo = {i:e for e,i in emo2id.items()}
df["emotion_id"] = df["emotion"].map(emo2id)

print("Emotion classes:", emo_classes)
df["emotion"].value_counts()


Emotion classes: ['ang', 'hap', 'neu', 'sad']


emotion
hap    2632
neu    1726
ang    1269
sad    1250
Name: count, dtype: int64

In [7]:
def activation_to_bin(a: float, cutpoints=(2.5, 3.5, 4.5)) -> int:
    c1, c2, c3 = cutpoints
    if a < c1: return 0
    if a < c2: return 1
    if a < c3: return 2
    return 3

dist_classes = ["low", "med", "high", "extreme"]
df["distress_bin"] = df["EmoAct"].astype(float).apply(lambda x: activation_to_bin(x, cfg.activation_bins))
df["distress_id"] = df["distress_bin"].astype(int)

df[["EmoAct","distress_bin"]].head(), df["distress_bin"].value_counts()


(     EmoAct  distress_bin
 0  2.333333             0
 1  2.666667             1
 2  2.666667             1
 3  3.000000             1
 4  3.333333             1,
 distress_bin
 1    3303
 2    2112
 0    1102
 3     360
 Name: count, dtype: int64)

In [8]:
sessions = sorted([s for s in df["session"].unique().tolist() if s.startswith("Ses")])
print("Detected sessions:", sessions)

def stratified_train_val_split(frame: pd.DataFrame, val_ratio: float = 0.1, seed: int = 42):
    rng = np.random.RandomState(seed)
    val_idx = []
    for c in frame["emotion_id"].unique():
        c_idx = frame.index[frame["emotion_id"] == c].values
        rng.shuffle(c_idx)
        n_val = max(1, int(val_ratio * len(c_idx)))
        val_idx.extend(c_idx[:n_val].tolist())
    val_idx = sorted(set(val_idx))
    val_df = frame.loc[val_idx].copy()
    tr_df = frame.drop(index=val_idx).copy()
    return tr_df.reset_index(drop=True), val_df.reset_index(drop=True)


Detected sessions: ['Ses01', 'Ses02', 'Ses03', 'Ses04', 'Ses05']


In [9]:
def resample_to_16k(wav: torch.Tensor, orig_sr: int, target_sr: int = 16000) -> torch.Tensor:
    if orig_sr == target_sr:
        return wav
    return torchaudio.functional.resample(wav, orig_sr, target_sr)

def truncate_or_pad(wav: torch.Tensor, target_len: int) -> torch.Tensor:
    if wav.numel() > target_len:
        return wav[:target_len]
    if wav.numel() < target_len:
        return F.pad(wav, (0, target_len - wav.numel()))
    return wav

def make_sliding_segments(wav: torch.Tensor, sr: int, seg_s: float, overlap: float):
    seg_len = int(seg_s * sr)
    hop = int(seg_len * (1.0 - overlap))
    hop = max(1, hop)
    L = wav.numel()
    if L <= seg_len:
        return [(wav, 0, L)]
    out = []
    for start in range(0, max(1, L - seg_len + 1), hop):
        end = start + seg_len
        out.append((wav[start:end], start, end))
        if end >= L:
            break
    return out


In [10]:
txt_tokenizer = DistilBertTokenizerFast.from_pretrained(cfg.text_name)
wav_processor = Wav2Vec2Processor.from_pretrained(cfg.wav2vec_name)


In [11]:
class HFIEMOCAPDataset(Dataset):
    def __init__(self, ds_hf, index_df: pd.DataFrame, cfg: Config):
        self.ds = ds_hf
        self.df = index_df.reset_index(drop=True)
        self.cfg = cfg

        # O(1) lookup map: hf_idx -> row dict
        # (Using dict of python scalars is fastest in __getitem__)
        self.row_by_hf_idx = {
            int(r["idx"]): r for r in self.df.to_dict(orient="records")
        }

        self.expanded = None
        if cfg.use_sliding_window:
            self.expanded = [(i, None) for i in range(len(self.df))]

    def __len__(self):
        return len(self.expanded) if self.expanded is not None else len(self.df)

    def _get_row(self, i: int):
        if self.expanded is None:
            row = self.df.iloc[i]
            return int(row["idx"]), None
        else:
            row_i, seg_i = self.expanded[i]
            row = self.df.iloc[row_i]
            return int(row["idx"]), seg_i

    def __getitem__(self, i: int):
        hf_idx, seg_i = self._get_row(i)

        # Fast O(1) row lookup
        row = self.row_by_hf_idx[hf_idx]

        item = self.ds[hf_idx]
        audio_obj = item["audio"]
        wav = torch.tensor(audio_obj["array"], dtype=torch.float32)
        sr = int(audio_obj["sampling_rate"])

        # mono
        if wav.ndim > 1:
            wav = wav.mean(dim=0)

        # resample
        wav = resample_to_16k(wav, sr, self.cfg.target_sr)

        seg_meta = (row["file"], 0, wav.numel())

        if self.cfg.use_sliding_window:
            segs = make_sliding_segments(wav, self.cfg.target_sr, self.cfg.segment_seconds, self.cfg.segment_overlap)
            k = i % len(segs)
            wav, s0, s1 = segs[k]
            seg_meta = (row["file"], int(s0), int(s1))

        # truncate/pad
        max_len = int(self.cfg.max_audio_seconds * self.cfg.target_sr)
        wav = truncate_or_pad(wav, max_len)

        return {
            "file": row["file"],
            "session": row["session"],
            "dialog_id": row["dialog_id"],
            "speaker": row["speaker"],
            "text": row["transcription"],
            "emotion_id": int(row["emotion_id"]),
            "emoact": float(row["EmoAct"]),
            "distress_id": int(row["distress_id"]),
            "wav": wav,
            "seg_meta": seg_meta,
        }


def collate_fn(batch: List[Dict[str, Any]], cfg: Config):
    texts = [b["text"] for b in batch]
    txt = txt_tokenizer(
        texts, padding=True, truncation=True, max_length=128, return_tensors="pt"
    )

    wavs = [b["wav"].numpy() for b in batch]
    wav_inputs = wav_processor(wavs, sampling_rate=cfg.target_sr, padding=True, return_tensors="pt")

    emo_y = torch.tensor([b["emotion_id"] for b in batch], dtype=torch.long)
    act_y = torch.tensor([b["emoact"] for b in batch], dtype=torch.float32)
    dist_y = torch.tensor([b["distress_id"] for b in batch], dtype=torch.long)

    return {
        "file": [b["file"] for b in batch],
        "session": [b["session"] for b in batch],
        "dialog_id": [b["dialog_id"] for b in batch],
        "speaker": [b["speaker"] for b in batch],
        "seg_meta": [b["seg_meta"] for b in batch],
        "text_input_ids": txt["input_ids"],
        "text_attn_mask": txt["attention_mask"],
        "audio_input_values": wav_inputs["input_values"],
        "audio_attn_mask": wav_inputs.get("attention_mask", None),
        "emo_y": emo_y,
        "act_y": act_y,
        "dist_y": dist_y,
    }


In [12]:
class DistressRepresentationBlock(nn.Module):
    def __init__(self, in_dim: int, use_stats: bool = True):
        super().__init__()
        self.use_stats = use_stats
        self.attn = nn.Sequential(
            nn.Linear(in_dim, in_dim),
            nn.Tanh(),
            nn.Linear(in_dim, 1),
        )

    def forward(self, Z: torch.Tensor, mask: Optional[torch.Tensor] = None):
        # Z: (B,T,D)
        scores = self.attn(Z).squeeze(-1)  # (B,T)
        if mask is not None:
            scores = scores.masked_fill(mask == 0, -1e9)
        alpha = torch.softmax(scores, dim=-1)  # (B,T)
        h = torch.bmm(alpha.unsqueeze(1), Z).squeeze(1)  # (B,D)

        if self.use_stats:
            if mask is None:
                mu = Z.mean(dim=1)
                sigma = Z.std(dim=1)
            else:
                m = mask.unsqueeze(-1)
                denom = m.sum(dim=1).clamp(min=1.0)
                mu = (Z * m).sum(dim=1) / denom
                var = (m * (Z - mu.unsqueeze(1))**2).sum(dim=1) / denom
                sigma = torch.sqrt(var + 1e-6)
            h = torch.cat([h, mu, sigma], dim=-1)

        return h, alpha

class CoralLayer(nn.Module):
    def __init__(self, in_dim: int, num_classes: int):
        super().__init__()
        self.num_classes = num_classes
        self.fc = nn.Linear(in_dim, num_classes - 1)
    def forward(self, x):
        return self.fc(x)

def coral_loss(logits: torch.Tensor, y: torch.Tensor, num_classes: int):
    # y in [0..K-1]; logits (B,K-1)
    B = y.shape[0]
    K = num_classes
    t = torch.zeros((B, K-1), device=logits.device, dtype=torch.float32)
    for k in range(K-1):
        t[:, k] = (y > k).float()
    return F.binary_cross_entropy_with_logits(logits, t)

class DERSXModel(nn.Module):
    def __init__(self, cfg: Config, num_emotions: int, num_distress_classes: int = 4):
        super().__init__()
        self.cfg = cfg
        self.num_distress_classes = num_distress_classes

        self.audio_enc = Wav2Vec2Model.from_pretrained(cfg.wav2vec_name)
        self.text_enc = DistilBertModel.from_pretrained(cfg.text_name)

        # (optional) freeze wav2vec2 for speed & parsimony
        for p in self.audio_enc.parameters():
            p.requires_grad = False

        # projections
        self.audio_proj = nn.Linear(self.audio_enc.config.hidden_size, cfg.proj_dim)
        self.text_proj  = nn.Linear(self.text_enc.config.hidden_size, cfg.proj_dim)

        # cross-modal attention (Q=audio, K/V=text)
        self.xattn = nn.MultiheadAttention(
            embed_dim=cfg.proj_dim,
            num_heads=cfg.num_heads,
            batch_first=True
        )

        fusion_dim = 2 * cfg.proj_dim if cfg.fusion_mode == "fusion" else cfg.proj_dim
        self.drb = DistressRepresentationBlock(fusion_dim, use_stats=cfg.drb_use_stats)

        drb_out_dim = fusion_dim * (3 if cfg.drb_use_stats else 1)

        self.dropout = nn.Dropout(0.2)
        self.emo_head = nn.Linear(drb_out_dim, num_emotions)

        if cfg.distress_target == "activation_regression":
            self.dist_head = nn.Linear(drb_out_dim, 1)
        elif cfg.distress_target == "activation_categorical":
            self.dist_head = nn.Linear(drb_out_dim, num_distress_classes)
        elif cfg.distress_target == "activation_ordinal":
            self.dist_head = CoralLayer(drb_out_dim, num_distress_classes)
        else:
            raise ValueError("Unknown distress_target")

    def forward(self, audio_input_values, audio_attn_mask, text_input_ids, text_attn_mask, return_attn=False):
        # encoders
        aud = self.audio_enc(input_values=audio_input_values, attention_mask=audio_attn_mask).last_hidden_state
        A = self.audio_proj(aud)  # (B,Ta,d)

        txt = self.text_enc(input_ids=text_input_ids, attention_mask=text_attn_mask).last_hidden_state
        T = self.text_proj(txt)  # (B,Tt,d)

        # Build text key padding mask for attention
        key_padding_mask = (text_attn_mask == 0)

        if self.cfg.fusion_mode == "fusion":
            A_text, attn_w = self.xattn(
                query=A, key=T, value=T,
                key_padding_mask=key_padding_mask,
                need_weights=True,
                average_attn_weights=False
            )
            Z = torch.cat([A, A_text], dim=-1)  # (B,Ta,2d)
        elif self.cfg.fusion_mode == "audio_only":
            attn_w = None
            Z = A
        else:  # text_only
            attn_w = None
            Z = T

        h, temporal_alpha = self.drb(Z, mask=None)
        h = self.dropout(h)

        emo_logits = self.emo_head(h)
        if self.cfg.distress_target == "activation_regression":
            dist_out = self.dist_head(h).squeeze(-1)
        else:
            dist_out = self.dist_head(h)

        if return_attn:
            return emo_logits, dist_out, attn_w, temporal_alpha
        return emo_logits, dist_out


In [13]:
def softmax_np(x: np.ndarray, axis=-1):
    e = np.exp(x - np.max(x, axis=axis, keepdims=True))
    return e / np.sum(e, axis=axis, keepdims=True)

def expected_calibration_error(probs: np.ndarray, y_true: np.ndarray, n_bins: int = 10) -> float:
    conf = probs.max(axis=1)
    pred = probs.argmax(axis=1)
    acc = (pred == y_true).astype(np.float32)
    bins = np.linspace(0, 1, n_bins + 1)
    ece = 0.0
    for i in range(n_bins):
        lo, hi = bins[i], bins[i+1]
        m = (conf >= lo) & (conf < hi) if i < n_bins - 1 else (conf >= lo) & (conf <= hi)
        if m.sum() == 0:
            continue
        ece += (m.mean()) * abs(acc[m].mean() - conf[m].mean())
    return float(ece)

def quadratic_weighted_kappa(y_true: np.ndarray, y_pred: np.ndarray, num_classes: int) -> float:
    O = confusion_matrix(y_true, y_pred, labels=list(range(num_classes))).astype(np.float64)
    N = O.sum()
    if N == 0:
        return 0.0
    W = np.zeros((num_classes, num_classes), dtype=np.float64)
    for i in range(num_classes):
        for j in range(num_classes):
            W[i, j] = ((i - j) ** 2) / ((num_classes - 1) ** 2)
    act_hist = O.sum(axis=1)
    pred_hist = O.sum(axis=0)
    E = np.outer(act_hist, pred_hist) / N
    num = (W * O).sum()
    den = (W * E).sum()
    return 1.0 - num / den if den > 0 else 0.0

def eval_emotion_metrics(y_true: np.ndarray, y_pred: np.ndarray) -> Dict[str, float]:
    return {
        "acc": float(accuracy_score(y_true, y_pred)),
        "macro_f1": float(f1_score(y_true, y_pred, average="macro")),
        "weighted_f1": float(f1_score(y_true, y_pred, average="weighted")),
        "uar": float(recall_score(y_true, y_pred, average="macro")),
    }

def eval_distress_metrics(cfg: Config, true_act: np.ndarray, pred_act: np.ndarray,
                         true_cls: Optional[np.ndarray]=None, pred_cls: Optional[np.ndarray]=None) -> Dict[str, float]:
    if cfg.distress_target == "activation_regression":
        return {"mae": float(np.mean(np.abs(pred_act - true_act)))}

    if true_cls is None or pred_cls is None:
        return {}

    out = {
        "acc": float(accuracy_score(true_cls, pred_cls)),
        "macro_f1": float(f1_score(true_cls, pred_cls, average="macro")),
        "weighted_f1": float(f1_score(true_cls, pred_cls, average="weighted")),
        "uar": float(recall_score(true_cls, pred_cls, average="macro")),
        "qwk": float(quadratic_weighted_kappa(true_cls, pred_cls, num_classes=4)),
    }
    # High/Extreme recall focus
    for cls, name in [(2, "high"), (3, "extreme")]:
        tp = np.sum((true_cls == cls) & (pred_cls == cls))
        fn = np.sum((true_cls == cls) & (pred_cls != cls))
        out[f"recall_{name}"] = float(tp / max(1, (tp + fn)))
    return out

def compute_losses(cfg: Config, emo_logits, dist_out, emo_y, act_y, dist_y, lambda_emo, lambda_dist):
    emo_loss = F.cross_entropy(emo_logits, emo_y)

    if cfg.distress_target == "activation_regression":
        dist_loss = F.smooth_l1_loss(dist_out, act_y)
    elif cfg.distress_target == "activation_categorical":
        dist_loss = F.cross_entropy(dist_out, dist_y)
    elif cfg.distress_target == "activation_ordinal":
        dist_loss = coral_loss(dist_out, dist_y, num_classes=4)
    else:
        raise ValueError("Unknown distress_target")

    total = lambda_emo * emo_loss + lambda_dist * dist_loss
    return total, emo_loss.detach(), dist_loss.detach()


In [14]:
from contextlib import nullcontext

def build_optimizer(model: nn.Module, cfg: Config):
    no_decay = ["bias", "LayerNorm.weight"]
    grouped = [
        {"params": [p for n,p in model.named_parameters() if p.requires_grad and not any(nd in n for nd in no_decay)],
         "weight_decay": cfg.weight_decay},
        {"params": [p for n,p in model.named_parameters() if p.requires_grad and any(nd in n for nd in no_decay)],
         "weight_decay": 0.0},
    ]
    return torch.optim.AdamW(grouped, lr=cfg.lr)

def run_epoch(model, loader, cfg: Config, optimizer=None, scheduler=None, train=True, lambda_emo=0.5, lambda_dist=0.5):
    model.train(train)
    scaler = torch.cuda.amp.GradScaler(enabled=(cfg.mixed_precision and DEVICE=="cuda"))
    amp_ctx = torch.cuda.amp.autocast if (cfg.mixed_precision and DEVICE=="cuda") else nullcontext

    total_loss = total_emo = total_dist = 0.0
    steps = 0

    emo_true, emo_pred, emo_logits_all = [], [], []
    act_true, act_pred = [], []
    dist_true, dist_pred, dist_raw = [], [], []

    files, dialog_ids = [], []

    for batch in tqdm(loader, leave=False):
        audio_input_values = batch["audio_input_values"].to(DEVICE)
        audio_attn_mask = batch["audio_attn_mask"]
        if audio_attn_mask is not None:
            audio_attn_mask = audio_attn_mask.to(DEVICE)

        text_input_ids = batch["text_input_ids"].to(DEVICE)
        text_attn_mask = batch["text_attn_mask"].to(DEVICE)

        emo_y = batch["emo_y"].to(DEVICE)
        act_y = batch["act_y"].to(DEVICE)
        dist_y = batch["dist_y"].to(DEVICE)

        with amp_ctx():
            emo_logits, dist_out = model(audio_input_values, audio_attn_mask, text_input_ids, text_attn_mask, return_attn=False)
            loss, emo_l, dist_l = compute_losses(cfg, emo_logits, dist_out, emo_y, act_y, dist_y, lambda_emo, lambda_dist)

        if train:
            optimizer.zero_grad(set_to_none=True)
            scaler.scale(loss).backward()
            if cfg.grad_clip is not None:
                scaler.unscale_(optimizer)
                nn.utils.clip_grad_norm_(model.parameters(), cfg.grad_clip)
            scaler.step(optimizer)
            scaler.update()
            if scheduler is not None:
                scheduler.step()

        total_loss += float(loss.detach().cpu())
        total_emo  += float(emo_l.cpu())
        total_dist += float(dist_l.cpu())
        steps += 1

        ep = torch.argmax(emo_logits, dim=-1)
        emo_true.append(emo_y.detach().cpu().numpy())
        emo_pred.append(ep.detach().cpu().numpy())
        emo_logits_all.append(emo_logits.detach().cpu().numpy())

        if cfg.distress_target == "activation_regression":
            act_true.append(act_y.detach().cpu().numpy())
            act_pred.append(dist_out.detach().cpu().numpy())
        elif cfg.distress_target == "activation_categorical":
            dp = torch.argmax(dist_out, dim=-1)
            dist_true.append(dist_y.detach().cpu().numpy())
            dist_pred.append(dp.detach().cpu().numpy())
            dist_raw.append(dist_out.detach().cpu().numpy())
        elif cfg.distress_target == "activation_ordinal":
            p = torch.sigmoid(dist_out)
            dp = torch.sum(p > 0.5, dim=1).long()
            dist_true.append(dist_y.detach().cpu().numpy())
            dist_pred.append(dp.detach().cpu().numpy())
            dist_raw.append(dist_out.detach().cpu().numpy())

        files.extend(batch["file"])
        dialog_ids.extend(batch["dialog_id"])

    out = {
        "loss": total_loss / max(1, steps),
        "emo_loss": total_emo / max(1, steps),
        "dist_loss": total_dist / max(1, steps),
        "emo_true": np.concatenate(emo_true),
        "emo_pred": np.concatenate(emo_pred),
        "emo_logits": np.concatenate(emo_logits_all),
        "file": files,
        "dialog_id": dialog_ids,
    }
    if cfg.distress_target == "activation_regression":
        out["act_true"] = np.concatenate(act_true)
        out["act_pred"] = np.concatenate(act_pred)
    else:
        out["dist_true"] = np.concatenate(dist_true)
        out["dist_pred"] = np.concatenate(dist_pred)
        out["dist_raw"]  = np.concatenate(dist_raw) if len(dist_raw) else None
    return out


In [15]:
def dialog_level_aggregate(pred_df: pd.DataFrame, cfg: Config):
    rows = []
    for dialog_id, g in pred_df.groupby("dialog_id"):
        # emotion: majority vote
        emo_true = int(pd.Series(g["emo_true"]).mode().iloc[0])
        emo_pred = int(pd.Series(g["emo_pred"]).mode().iloc[0])
        row = {"dialog_id": dialog_id, "emo_true": emo_true, "emo_pred": emo_pred}

        if cfg.distress_target == "activation_regression":
            row["act_true"] = float(g["act_true"].mean())
            row["act_pred"] = float(g["act_pred"].mean())
        else:
            row["dist_true"] = int(pd.Series(g["dist_true"]).mode().iloc[0])
            row["dist_pred"] = int(pd.Series(g["dist_pred"]).mode().iloc[0])

        rows.append(row)
    return pd.DataFrame(rows)


In [16]:
def fit_one_setting(train_df, val_df, test_df, lambda_emo, lambda_dist, cfg: Config):
    train_ds = HFIEMOCAPDataset(ds, train_df, cfg)
    val_ds   = HFIEMOCAPDataset(ds, val_df, cfg)
    test_ds  = HFIEMOCAPDataset(ds, test_df, cfg)

    train_loader = DataLoader(train_ds, batch_size=cfg.batch_size, shuffle=True,
                              collate_fn=lambda b: collate_fn(b, cfg),
                              num_workers=0, pin_memory=(DEVICE=="cuda"))
    val_loader   = DataLoader(val_ds, batch_size=cfg.batch_size, shuffle=False,
                              collate_fn=lambda b: collate_fn(b, cfg),
                              num_workers=0, pin_memory=(DEVICE=="cuda"))
    test_loader  = DataLoader(test_ds, batch_size=cfg.batch_size, shuffle=False,
                              collate_fn=lambda b: collate_fn(b, cfg),
                              num_workers=0, pin_memory=(DEVICE=="cuda"))

    # smoke test
    _ = next(iter(train_loader))
    print("First batch OK")

    model = DERSXModel(cfg, num_emotions=len(emo_classes), num_distress_classes=4).to(DEVICE)
    optimizer = build_optimizer(model, cfg)

    total_steps = len(train_loader) * cfg.num_epochs
    warmup_steps = int(cfg.warmup_ratio * total_steps)
    scheduler = get_linear_schedule_with_warmup(optimizer, warmup_steps, total_steps)

    best_val = -1e9
    best_state = None
    history = []

    for ep in range(1, cfg.num_epochs + 1):
        tr = run_epoch(model, train_loader, cfg, optimizer, scheduler, train=True,
                       lambda_emo=lambda_emo, lambda_dist=lambda_dist)
        va = run_epoch(model, val_loader, cfg, None, None, train=False,
                       lambda_emo=lambda_emo, lambda_dist=lambda_dist)

        emo_m = eval_emotion_metrics(va["emo_true"], va["emo_pred"])
        sel = emo_m["macro_f1"]

        history.append({
            "epoch": ep,
            "train_loss": tr["loss"],
            "val_loss": va["loss"],
            "val_emo_acc": emo_m["acc"],
            "val_emo_macro_f1": emo_m["macro_f1"],
        })
        print(f"epoch {ep} | train {tr['loss']:.4f} | val {va['loss']:.4f} | val emo macroF1 {emo_m['macro_f1']:.4f}")

        if sel > best_val:
            best_val = sel
            best_state = {k: v.cpu() for k,v in model.state_dict().items()}

    if best_state is not None:
        model.load_state_dict(best_state)

    te = run_epoch(model, test_loader, cfg, None, None, train=False,
                   lambda_emo=lambda_emo, lambda_dist=lambda_dist)
    return model, history, te


all_fold_rows = []
all_pred_rows = []

run_tag = time.strftime("%Y%m%d_%H%M%S")
run_dir = os.path.join(cfg.out_dir, f"run_{run_tag}")
os.makedirs(run_dir, exist_ok=True)

for held_out in sessions:
    fold_name = f"LOSO_{held_out}"
    test_df = df[df["session"] == held_out].reset_index(drop=True)
    train_full = df[df["session"] != held_out].reset_index(drop=True)
    train_df, val_df = stratified_train_val_split(train_full, val_ratio=0.1, seed=42)

    best_score = -1e9
    best_pack = None
    best_lams = None

    for lam in cfg.lambda_grid:
        lambda_emo = lam
        lambda_dist = 1.0 - lam
        print(f"\n=== {fold_name} | λ_emo={lambda_emo:.1f} λ_dist={lambda_dist:.1f} ===")
        model, hist, te = fit_one_setting(train_df, val_df, test_df, lambda_emo, lambda_dist, cfg)

        emo_m = eval_emotion_metrics(te["emo_true"], te["emo_pred"])
        score = emo_m["macro_f1"]
        if score > best_score:
            best_score = score
            best_pack = (model, hist, te)
            best_lams = (lambda_emo, lambda_dist)

    model, hist, te = best_pack
    lambda_emo, lambda_dist = best_lams
    print(f">>> Best for {fold_name}: λ_emo={lambda_emo:.1f}, λ_dist={lambda_dist:.1f} | test emo macroF1={best_score:.4f}")

    # ===== Metrics (utterance-level) =====
    emo_m = eval_emotion_metrics(te["emo_true"], te["emo_pred"])
    emo_probs = softmax_np(te["emo_logits"], axis=1)
    emo_ece = expected_calibration_error(emo_probs, te["emo_true"], n_bins=10)

    # Cohen kappa: system vs majority label (major_emotion mapped)
    emo_kappa = float(cohen_kappa_score(te["emo_true"], te["emo_pred"]))

    fold_row = {
        "fold": fold_name,
        "held_out": held_out,
        "lambda_emo": lambda_emo,
        "lambda_dist": lambda_dist,
        "emo_acc": emo_m["acc"],
        "emo_macro_f1": emo_m["macro_f1"],
        "emo_weighted_f1": emo_m["weighted_f1"],
        "emo_uar": emo_m["uar"],
        "emo_ece": emo_ece,
        "emo_cohen_kappa_sys_vs_majority": emo_kappa,
    }

    # Distress metrics
    if cfg.distress_target == "activation_regression":
        dist_m = eval_distress_metrics(cfg, te["act_true"], te["act_pred"])
        fold_row.update({f"dist_{k}": v for k,v in dist_m.items()})
    else:
        dist_m = eval_distress_metrics(cfg, te.get("act_true", None), te.get("act_pred", None),
                                       te["dist_true"], te["dist_pred"])
        fold_row.update({f"dist_{k}": v for k,v in dist_m.items()})
        if cfg.distress_target == "activation_categorical" and te.get("dist_raw") is not None:
            dist_probs = softmax_np(te["dist_raw"], axis=1)
            fold_row["dist_ece"] = expected_calibration_error(dist_probs, te["dist_true"], n_bins=10)

    all_fold_rows.append(fold_row)

    # ===== Save utterance predictions =====
    pred_df = pd.DataFrame({
        "fold": fold_name,
        "file": te["file"],
        "dialog_id": te["dialog_id"],
        "emo_true": te["emo_true"],
        "emo_pred": te["emo_pred"],
    })
    if cfg.distress_target == "activation_regression":
        pred_df["act_true"] = te["act_true"]
        pred_df["act_pred"] = te["act_pred"]
    else:
        pred_df["dist_true"] = te["dist_true"]
        pred_df["dist_pred"] = te["dist_pred"]

    all_pred_rows.append(pred_df)

# Save outputs
fold_results = pd.DataFrame(all_fold_rows)
preds = pd.concat(all_pred_rows, axis=0).reset_index(drop=True)

fold_results.to_csv(os.path.join(run_dir, "fold_results.csv"), index=False)
preds.to_csv(os.path.join(run_dir, "utterance_predictions.csv"), index=False)
with open(os.path.join(run_dir, "config.json"), "w") as f:
    json.dump(asdict(cfg), f, indent=2)

print("Saved run to:", run_dir)
fold_results



=== LOSO_Ses01 | λ_emo=0.3 λ_dist=0.7 ===
First batch OK


Some weights of Wav2Vec2Model were not initialized from the model checkpoint at facebook/wav2vec2-base-960h and are newly initialized: ['masked_spec_embed']
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.


  0%|          | 0/832 [00:00<?, ?it/s]

  0%|          | 0/92 [00:00<?, ?it/s]

epoch 1 | train nan | val 2.2340 | val emo macroF1 0.1687


...


=== LOSO_Ses02 | λ_emo=0.5 λ_dist=0.5 ===
First batch OK


Some weights of Wav2Vec2Model were not initialized from the model checkpoint at facebook/wav2vec2-base-960h and are newly initialized: ['masked_spec_embed']
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.


  0%|          | 0/846 [00:00<?, ?it/s]

  0%|          | 0/94 [00:00<?, ?it/s]

KeyboardInterrupt: 

In [ ]:
def plot_confusion(cm: np.ndarray, labels: List[str], title: str, save_path: Optional[str]=None):
    plt.figure(figsize=(6, 5))
    plt.imshow(cm, interpolation="nearest")
    plt.title(title)
    plt.colorbar()
    ticks = np.arange(len(labels))
    plt.xticks(ticks, labels, rotation=45, ha="right")
    plt.yticks(ticks, labels)

    thr = cm.max() / 2.0 if cm.max() > 0 else 0.5
    for i in range(cm.shape[0]):
        for j in range(cm.shape[1]):
            plt.text(j, i, int(cm[i, j]), ha="center", color="white" if cm[i,j] > thr else "black")

    plt.ylabel("True")
    plt.xlabel("Predicted")
    plt.tight_layout()
    if save_path:
        plt.savefig(save_path, dpi=200, bbox_inches="tight")
    plt.show()

# Emotion CM overall
emo_cm = confusion_matrix(preds["emo_true"], preds["emo_pred"], labels=list(range(len(emo_classes))))
plot_confusion(emo_cm, [id2emo[i] for i in range(len(emo_classes))],
               "Emotion Confusion Matrix (All folds)",
               save_path=os.path.join(run_dir, "emotion_confusion.png"))

# Distress CM overall (if categorical/ordinal)
if cfg.distress_target != "activation_regression":
    dist_cm = confusion_matrix(preds["dist_true"], preds["dist_pred"], labels=[0,1,2,3])
    plot_confusion(dist_cm, dist_classes,
                   "Distress/Activation Confusion Matrix (All folds)",
                   save_path=os.path.join(run_dir, "distress_confusion.png"))

    y_true = preds["dist_true"].values
    y_pred = preds["dist_pred"].values
    for cls, name in [(2, "high"), (3, "extreme")]:
        tp = np.sum((y_true == cls) & (y_pred == cls))
        fn = np.sum((y_true == cls) & (y_pred != cls))
        print(f"Recall({name}): {tp / max(1, tp+fn):.4f}")


In [ ]:
dlg = dialog_level_aggregate(preds, cfg)

dlg_emo = eval_emotion_metrics(dlg["emo_true"].values, dlg["emo_pred"].values)
print("Dialog-level emotion:", dlg_emo)

if cfg.distress_target == "activation_regression":
    mae = float(np.mean(np.abs(dlg["act_pred"].values - dlg["act_true"].values)))
    print("Dialog-level activation MAE:", mae)
else:
    dmet = eval_distress_metrics(cfg, None, None, dlg["dist_true"].values, dlg["dist_pred"].values)
    print("Dialog-level distress:", dmet)


In [ ]:
def reliability_diagram(probs: np.ndarray, y_true: np.ndarray, n_bins: int = 10, title: str = "", save_path: Optional[str]=None):
    conf = probs.max(axis=1)
    pred = probs.argmax(axis=1)
    acc = (pred == y_true).astype(np.float32)

    bins = np.linspace(0, 1, n_bins + 1)
    xs, ys = [], []
    for i in range(n_bins):
        lo, hi = bins[i], bins[i+1]
        m = (conf >= lo) & (conf < hi) if i < n_bins-1 else (conf >= lo) & (conf <= hi)
        if m.sum() == 0:
            continue
        xs.append(conf[m].mean())
        ys.append(acc[m].mean())

    plt.figure(figsize=(6,5))
    plt.plot([0,1],[0,1])
    plt.scatter(xs, ys)
    plt.xlabel("Confidence")
    plt.ylabel("Accuracy")
    plt.title(title)
    plt.tight_layout()
    if save_path:
        plt.savefig(save_path, dpi=200, bbox_inches="tight")
    plt.show()

print("Mean emotion ECE:", float(fold_results["emo_ece"].mean()))


In [ ]:
def show_attention_example(model: DERSXModel, sample_idx: int):
    model.eval()
    sample_row = df.iloc[sample_idx]
    ex = HFIEMOCAPDataset(ds, df.iloc[[sample_idx]], cfg)[0]
    batch = collate_fn([ex], cfg)

    audio = batch["audio_input_values"].to(DEVICE)
    amask = batch["audio_attn_mask"]
    if amask is not None:
        amask = amask.to(DEVICE)

    tids = batch["text_input_ids"].to(DEVICE)
    tmask = batch["text_attn_mask"].to(DEVICE)

    with torch.no_grad():
        emo_logits, dist_out, xattn_w, temporal_alpha = model(audio, amask, tids, tmask, return_attn=True)

    tokens = txt_tokenizer.convert_ids_to_tokens(batch["text_input_ids"][0].tolist())
    tokens = tokens[:40]

    print("FILE:", ex["file"])
    print("TRUE emotion:", id2emo[ex["emotion_id"]])
    print("PRED emotion:", id2emo[int(torch.argmax(emo_logits, dim=-1).cpu().item())])

    if cfg.distress_target == "activation_regression":
        print("TRUE EmoAct:", ex["emoact"])
        print("PRED EmoAct:", float(dist_out.cpu().item()))
    else:
        if cfg.distress_target == "activation_categorical":
            dp = int(torch.argmax(dist_out, dim=-1).cpu().item())
        else:
            p = torch.sigmoid(dist_out)
            dp = int(torch.sum(p > 0.5, dim=1).cpu().item())
        print("TRUE distress:", dist_classes[ex["distress_id"]])
        print("PRED distress:", dist_classes[dp])

    if xattn_w is not None:
        w = xattn_w[0].detach().cpu().numpy()  # (heads, Ta, Tt)
        w_mean = w.mean(axis=0)               # (Ta, Tt)
        token_imp = w_mean.mean(axis=0)[:len(tokens)]
        token_imp = token_imp / (token_imp.sum() + 1e-9)

        plt.figure(figsize=(10,3))
        plt.bar(range(len(tokens)), token_imp)
        plt.xticks(range(len(tokens)), tokens, rotation=60, ha="right")
        plt.title("Cross-modal attention token importance (avg over heads & audio frames)")
        plt.tight_layout()
        plt.show()

    temp = temporal_alpha[0].detach().cpu().numpy()
    temp = temp / (temp.sum() + 1e-9)
    plt.figure(figsize=(10,3))
    plt.plot(temp)
    plt.title("DRB temporal attention weights (audio frames)")
    plt.tight_layout()
    plt.show()

print("To run this, keep a trained `model` from a fold in scope, then call:")
print("show_attention_example(model, sample_idx=0)")


In [ ]:
try:
    from captum.attr import IntegratedGradients
    CAPTUM_OK = True
except Exception as e:
    CAPTUM_OK = False
    print("Captum not available:", e)

class IGWrapper(nn.Module):
    def __init__(self, model: DERSXModel):
        super().__init__()
        self.model = model
    def forward(self, audio_input_values, text_input_ids, text_attn_mask, target_class):
        emo_logits, _ = self.model(audio_input_values, None, text_input_ids, text_attn_mask, return_attn=False)
        idx = target_class.view(-1,1)
        return torch.gather(emo_logits, 1, idx).squeeze(1)

def run_ig(model: DERSXModel, sample_idx: int, target_emo_id: int, steps: int = 32):
    if not CAPTUM_OK:
        print("Install captum first.")
        return
    model.eval()
    ex = HFIEMOCAPDataset(ds, df.iloc[[sample_idx]], cfg)[0]
    batch = collate_fn([ex], cfg)

    audio = batch["audio_input_values"].to(DEVICE)
    tids = batch["text_input_ids"].to(DEVICE)
    tmask = batch["text_attn_mask"].to(DEVICE)
    target = torch.tensor([target_emo_id], device=DEVICE)

    wrapper = IGWrapper(model).to(DEVICE)
    ig = IntegratedGradients(wrapper)

    baseline = torch.zeros_like(audio)
    attr, delta = ig.attribute(
        inputs=(audio, tids, tmask, target),
        baselines=(baseline, tids, tmask, target),
        n_steps=steps,
        return_convergence_delta=True
    )

    wav_attr = attr[0].detach().cpu().numpy()[0]
    wav_attr = np.abs(wav_attr)
    wav_attr = wav_attr / (wav_attr.max() + 1e-9)

    plt.figure(figsize=(12,3))
    plt.plot(wav_attr)
    plt.title("Integrated Gradients |waveform| attribution (normalized)")
    plt.tight_layout()
    plt.show()
    print("Convergence delta:", float(delta.detach().cpu().item()))

print("After training, call: run_ig(model, sample_idx=0, target_emo_id=emo2id['neu'])")


In [ ]:
def mean_std(series: pd.Series):
    return float(series.mean()), float(series.std(ddof=1))

def per_class_f1(y_true: np.ndarray, y_pred: np.ndarray, class_names: List[str]):
    f1s = f1_score(y_true, y_pred, average=None, labels=list(range(len(class_names))))
    return pd.DataFrame({"class": class_names, "f1": f1s})

def save_latex_table(df: pd.DataFrame, path: str, caption: str = "", label: str = ""):
    tex = df.to_latex(index=False, float_format=lambda x: f"{x:.4f}")
    if caption:
        tex = tex.replace("\\begin{table}", "\\begin{table}\n\\caption{" + caption + "}")
    if label:
        tex = tex.replace("\\caption{"+caption+"}", "\\caption{"+caption+"}\\label{"+label+"}")
    with open(path, "w") as f:
        f.write(tex)


In [ ]:
def plot_bar_with_error(means, stds, labels, title, ylabel, save_path=None):
    x = np.arange(len(labels))
    plt.figure(figsize=(10, 4))
    plt.bar(x, means, yerr=stds, capsize=4)
    plt.xticks(x, labels, rotation=35, ha="right")
    plt.title(title)
    plt.ylabel(ylabel)
    plt.tight_layout()
    if save_path:
        plt.savefig(save_path, dpi=250, bbox_inches="tight")
    plt.show()

def plot_training_curves(histories: List[dict], title="Training curves", save_path=None):
    # histories: list of {"fold":..., "history": [ {epoch, train_loss, val_loss,...}, ... ] }
    plt.figure(figsize=(10,4))
    for h in histories:
        dfh = pd.DataFrame(h["history"])
        plt.plot(dfh["epoch"], dfh["val_loss"], label=f"{h['fold']}-val", alpha=0.6)
    plt.title(title)
    plt.xlabel("Epoch")
    plt.ylabel("Val loss")
    plt.tight_layout()
    if save_path:
        plt.savefig(save_path, dpi=250, bbox_inches="tight")
    plt.show()

def reliability_diagram(probs: np.ndarray, y_true: np.ndarray, n_bins: int = 10, title: str = "", save_path=None):
    conf = probs.max(axis=1)
    pred = probs.argmax(axis=1)
    acc = (pred == y_true).astype(np.float32)

    bins = np.linspace(0, 1, n_bins + 1)
    xs, ys = [], []
    for i in range(n_bins):
        lo, hi = bins[i], bins[i+1]
        m = (conf >= lo) & (conf < hi) if i < n_bins-1 else (conf >= lo) & (conf <= hi)
        if m.sum() == 0:
            continue
        xs.append(conf[m].mean())
        ys.append(acc[m].mean())

    plt.figure(figsize=(5,5))
    plt.plot([0,1],[0,1])
    plt.scatter(xs, ys)
    plt.xlabel("Confidence")
    plt.ylabel("Accuracy")
    plt.title(title)
    plt.tight_layout()
    if save_path:
        plt.savefig(save_path, dpi=250, bbox_inches="tight")
    plt.show()


In [ ]:
def run_loso_experiment(cfg: Config, exp_name: str, base_out_dir: str):
    run_tag = time.strftime("%Y%m%d_%H%M%S")
    run_dir = os.path.join(base_out_dir, f"{exp_name}_{run_tag}")
    os.makedirs(run_dir, exist_ok=True)

    fold_rows = []
    pred_rows = []
    histories = []
    best_models = {}  # optional: keep model per fold for explainability

    for held_out in sessions:
        fold_name = f"LOSO_{held_out}"
        test_df = df[df["session"] == held_out].reset_index(drop=True)
        train_full = df[df["session"] != held_out].reset_index(drop=True)
        train_df, val_df = stratified_train_val_split(train_full, val_ratio=0.1, seed=42)

        best_score = -1e9
        best_pack = None
        best_lams = None

        for lam in cfg.lambda_grid:
            lambda_emo = lam
            lambda_dist = 1.0 - lam
            print(f"\n[{exp_name}] {fold_name} | trying λ_emo={lambda_emo:.1f}, λ_dist={lambda_dist:.1f}")

            model, hist, te = fit_one_setting(train_df, val_df, test_df, lambda_emo, lambda_dist, cfg)
            emo_m = eval_emotion_metrics(te["emo_true"], te["emo_pred"])
            score = emo_m["macro_f1"]

            if score > best_score:
                best_score = score
                best_pack = (model, hist, te)
                best_lams = (lambda_emo, lambda_dist)

        model, hist, te = best_pack
        lambda_emo, lambda_dist = best_lams
        best_models[fold_name] = model

        # store history
        histories.append({"fold": fold_name, "history": hist})

        # ===== utterance-level metrics =====
        emo_m = eval_emotion_metrics(te["emo_true"], te["emo_pred"])
        emo_probs = softmax_np(te["emo_logits"], axis=1)
        emo_ece = expected_calibration_error(emo_probs, te["emo_true"], n_bins=10)
        emo_kappa = float(cohen_kappa_score(te["emo_true"], te["emo_pred"]))

        fold_row = {
            "exp": exp_name,
            "fold": fold_name,
            "held_out": held_out,
            "lambda_emo": lambda_emo,
            "lambda_dist": lambda_dist,
            "emo_acc": emo_m["acc"],
            "emo_macro_f1": emo_m["macro_f1"],
            "emo_weighted_f1": emo_m["weighted_f1"],
            "emo_uar": emo_m["uar"],
            "emo_ece": emo_ece,
            "emo_cohen_kappa_sys_vs_majority": emo_kappa,
        }

        if cfg.distress_target == "activation_regression":
            dist_m = eval_distress_metrics(cfg, te["act_true"], te["act_pred"])
            fold_row.update({f"dist_{k}": v for k,v in dist_m.items()})
        else:
            dist_m = eval_distress_metrics(cfg, None, None, te["dist_true"], te["dist_pred"])
            fold_row.update({f"dist_{k}": v for k,v in dist_m.items()})
            if cfg.distress_target == "activation_categorical" and te.get("dist_raw") is not None:
                dist_probs = softmax_np(te["dist_raw"], axis=1)
                fold_row["dist_ece"] = expected_calibration_error(dist_probs, te["dist_true"], n_bins=10)

        fold_rows.append(fold_row)

        # ===== save utterance preds =====
        pred_df = pd.DataFrame({
            "exp": exp_name,
            "fold": fold_name,
            "file": te["file"],
            "dialog_id": te["dialog_id"],
            "emo_true": te["emo_true"],
            "emo_pred": te["emo_pred"],
        })
        if cfg.distress_target == "activation_regression":
            pred_df["act_true"] = te["act_true"]
            pred_df["act_pred"] = te["act_pred"]
        else:
            pred_df["dist_true"] = te["dist_true"]
            pred_df["dist_pred"] = te["dist_pred"]
        pred_rows.append(pred_df)

        # ===== reliability plot per fold (emotion) =====
        reliability_diagram(
            emo_probs, te["emo_true"],
            title=f"{exp_name} {fold_name} Reliability (Emotion)",
            save_path=os.path.join(run_dir, f"reliability_{fold_name}.png")
        )

    fold_results = pd.DataFrame(fold_rows)
    preds = pd.concat(pred_rows, axis=0).reset_index(drop=True)

    # Save
    fold_results.to_csv(os.path.join(run_dir, "fold_results.csv"), index=False)
    preds.to_csv(os.path.join(run_dir, "utterance_predictions.csv"), index=False)
    with open(os.path.join(run_dir, "config.json"), "w") as f:
        json.dump(asdict(cfg), f, indent=2)

    return {
        "exp_name": exp_name,
        "run_dir": run_dir,
        "cfg": cfg,
        "fold_results": fold_results,
        "preds": preds,
        "histories": histories,
        "best_models": best_models,
    }


In [ ]:
def make_paper_tables(bundle):
    run_dir = bundle["run_dir"]
    fold_results = bundle["fold_results"]
    preds = bundle["preds"]
    exp_name = bundle["exp_name"]

    # === Mean ± Std across folds ===
    metrics_cols = [c for c in fold_results.columns if c not in ["exp","fold","held_out"]]
    summary_rows = []
    for c in metrics_cols:
        if pd.api.types.is_numeric_dtype(fold_results[c]):
            m, s = mean_std(fold_results[c])
            summary_rows.append({"metric": c, "mean": m, "std": s})
    summary_df = pd.DataFrame(summary_rows).sort_values("metric")

    summary_df.to_csv(os.path.join(run_dir, "paper_summary_mean_std.csv"), index=False)
    save_latex_table(summary_df, os.path.join(run_dir, "paper_summary_mean_std.tex"),
                     caption=f"{exp_name}: Mean±Std over LOSO folds", label=f"tab:{exp_name}_summary")

    # === Per-class F1 (emotion) utterance-level ===
    class_f1_df = per_class_f1(preds["emo_true"].values, preds["emo_pred"].values, emo_classes)
    class_f1_df.to_csv(os.path.join(run_dir, "emotion_per_class_f1.csv"), index=False)
    save_latex_table(class_f1_df, os.path.join(run_dir, "emotion_per_class_f1.tex"),
                     caption=f"{exp_name}: Per-class F1 (Emotion)", label=f"tab:{exp_name}_emo_class_f1")

    # === Dialog/call-level aggregation ===
    dlg = dialog_level_aggregate(preds, bundle["cfg"])
    dlg_emo = eval_emotion_metrics(dlg["emo_true"].values, dlg["emo_pred"].values)

    dlg_row = {"exp": exp_name, **{f"dlg_emo_{k}": v for k,v in dlg_emo.items()}}

    if bundle["cfg"].distress_target == "activation_regression":
        dlg_row["dlg_act_mae"] = float(np.mean(np.abs(dlg["act_pred"].values - dlg["act_true"].values)))
    else:
        dlg_dist = eval_distress_metrics(bundle["cfg"], None, None, dlg["dist_true"].values, dlg["dist_pred"].values)
        dlg_row.update({f"dlg_dist_{k}": v for k,v in dlg_dist.items()})

    dlg_df = pd.DataFrame([dlg_row])
    dlg_df.to_csv(os.path.join(run_dir, "dialog_level_metrics.csv"), index=False)
    save_latex_table(dlg_df, os.path.join(run_dir, "dialog_level_metrics.tex"),
                     caption=f"{exp_name}: Dialog-level (call-level) metrics", label=f"tab:{exp_name}_dlg")

    return summary_df, class_f1_df, dlg_df

print("Ready: call make_paper_tables(bundle) after running an experiment.")


In [ ]:
def make_result_graphs(bundle):
    run_dir = bundle["run_dir"]
    fold_results = bundle["fold_results"]
    preds = bundle["preds"]
    histories = bundle["histories"]
    exp_name = bundle["exp_name"]

    # training curves
    plot_training_curves(histories, title=f"{exp_name}: Val-loss curves",
                         save_path=os.path.join(run_dir, "training_curves_val_loss.png"))

    # fold-wise bar: emotion macro-F1
    labels = fold_results["fold"].tolist()
    means = fold_results["emo_macro_f1"].values
    stds = np.zeros_like(means)
    plot_bar_with_error(means, stds, labels, f"{exp_name}: Emotion Macro-F1 per fold", "Macro-F1",
                        save_path=os.path.join(run_dir, "emo_macro_f1_per_fold.png"))

    # confusion matrix overall emotion
    emo_cm = confusion_matrix(preds["emo_true"], preds["emo_pred"], labels=list(range(len(emo_classes))))
    plot_confusion(emo_cm, emo_classes, f"{exp_name}: Emotion Confusion (All folds)",
                   save_path=os.path.join(run_dir, "emotion_confusion_all.png"))

    # distress confusion if categorical/ordinal
    if bundle["cfg"].distress_target != "activation_regression":
        dist_cm = confusion_matrix(preds["dist_true"], preds["dist_pred"], labels=[0,1,2,3])
        plot_confusion(dist_cm, dist_classes, f"{exp_name}: Distress Confusion (All folds)",
                       save_path=os.path.join(run_dir, "distress_confusion_all.png"))

print("Ready: call make_result_graphs(bundle) after running an experiment.")


In [ ]:
from copy import deepcopy

def override_cfg(cfg: Config, overrides: Dict[str, Any]) -> Config:
    new = deepcopy(cfg)
    for k,v in overrides.items():
        if not hasattr(new, k):
            raise ValueError(f"Config has no field named '{k}'")
        setattr(new, k, v)
    return new

# ---- Define ablations ----
ablations = [
    ("A_fusion_default", {}),
    ("B_audio_only", {"fusion_mode": "audio_only"}),
    ("C_text_only", {"fusion_mode": "text_only"}),

    # DRB ablation
    ("D_no_drb_stats", {"drb_use_stats": False}),

    # Segmentation ablation (emergency-style)
    ("E_sliding_window", {"use_sliding_window": True, "segment_seconds": 8.0, "segment_overlap": 0.5}),

    # Distress head ablations (same “distress intensity” target from EmoAct)
    ("F_distress_categorical", {"distress_target": "activation_categorical"}),
    ("G_distress_ordinal_coral", {"distress_target": "activation_ordinal"}),
]

ablation_bundles = []
for name, ov in ablations:
    cfg_i = override_cfg(cfg, ov)
    print("\n==============================")
    print("Running ablation:", name)
    print("Overrides:", ov)
    bundle = run_loso_experiment(cfg_i, exp_name=name, base_out_dir=cfg.out_dir)
    make_paper_tables(bundle)
    make_result_graphs(bundle)
    ablation_bundles.append(bundle)

print("\nAll ablations completed.")


In [ ]:
# Collect mean±std of key metrics for each ablation
rows = []
for b in ablation_bundles:
    fr = b["fold_results"]
    row = {"exp": b["exp_name"]}
    row["emo_macro_f1_mean"] = fr["emo_macro_f1"].mean()
    row["emo_macro_f1_std"]  = fr["emo_macro_f1"].std(ddof=1)
    row["emo_uar_mean"]      = fr["emo_uar"].mean()
    row["emo_uar_std"]       = fr["emo_uar"].std(ddof=1)

    if b["cfg"].distress_target == "activation_regression":
        row["dist_mae_mean"] = fr["dist_mae"].mean()
        row["dist_mae_std"]  = fr["dist_mae"].std(ddof=1)
    else:
        # pick a primary distress metric
        row["dist_qwk_mean"] = fr["dist_qwk"].mean()
        row["dist_qwk_std"]  = fr["dist_qwk"].std(ddof=1)
        row["dist_recall_high_mean"] = fr.get("dist_recall_high", pd.Series([np.nan]*len(fr))).mean()
        row["dist_recall_extreme_mean"] = fr.get("dist_recall_extreme", pd.Series([np.nan]*len(fr))).mean()

    rows.append(row)

ablation_summary = pd.DataFrame(rows).sort_values("emo_macro_f1_mean", ascending=False)
ablation_summary


In [ ]:
# Save & make a simple publication bar chart: Emotion macro-F1 across ablations
out_path = os.path.join(cfg.out_dir, "ablation_summary.csv")
ablation_summary.to_csv(out_path, index=False)
print("Saved:", out_path)

labels = ablation_summary["exp"].tolist()
means = ablation_summary["emo_macro_f1_mean"].values
stds  = ablation_summary["emo_macro_f1_std"].values

plot_bar_with_error(means, stds, labels, "Ablation: Emotion Macro-F1 (Mean±Std over LOSO)", "Macro-F1",
                    save_path=os.path.join(cfg.out_dir, "ablation_emo_macro_f1.png"))
